# 1 — States and gates: a qubit is a unit vector

## What you will learn

By the end of this notebook you will be able to say, precisely and without hand-waving:

1. **What a qubit is.** Not "0 and 1 at the same time" — a *unit vector in $\mathbb{C}^2$*. You already know what that is.
2. **What Dirac notation means.** $|0\rangle$, $|1\rangle$, $|\psi\rangle$, $\langle\psi|$ — defined from scratch, in terms of column vectors and conjugate transposes.
3. **What a gate is.** A $2\times2$ matrix multiplying that vector. Nothing more exotic.
4. **What measurement is.** Where the probabilities come from (the *Born rule*), and why measuring is the one thing that destroys information.
5. **Why the amplitudes are complex numbers.** This is the real payoff. Complex phases are invisible to any single measurement — and then they decide the answer. That is *interference*, and it is the whole reason quantum computing exists.
6. **How two qubits combine.** The tensor product, and why we store the state as an array of shape `(2, 2)` rather than a flat vector of length 4.

**What you need to bring:** linear algebra (vectors, matrices, complex numbers, conjugate transposes) and basic NumPy. **No physics.** Every physical idea is introduced here at the point where it is first needed.

Everything below runs against `qsim`, this project's simulator. If a claim in the prose is checkable, there is a code cell right after it that checks it.

## A qubit is a unit vector in $\mathbb{C}^2$

A classical bit is an element of $\{0, 1\}$. A **qubit** is a unit vector in the two-dimensional complex vector space $\mathbb{C}^2$:

$$\psi = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \qquad \alpha, \beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

That is the entire definition. The two complex numbers $\alpha$ and $\beta$ are called **amplitudes** — one per classical outcome. They are *not* probabilities: they are complex, they can be negative, and they can cancel each other out. You get probabilities from them by squaring their magnitudes:

$$\Pr[\text{measure } 0] = |\alpha|^2, \qquad \Pr[\text{measure } 1] = |\beta|^2.$$

The normalization condition $|\alpha|^2 + |\beta|^2 = 1$ is now readable: it just says the two probabilities sum to 1. "Unit vector" and "these are probabilities" are the same statement.

A word about the phrase you have certainly heard, *"a qubit is 0 and 1 at the same time"*. It is not wrong so much as useless — it suggests a qubit is a coin mid-flip, hiding a value we merely don't know yet. It isn't. A vector with $\alpha = \tfrac{1}{\sqrt2}$, $\beta = \tfrac{1}{\sqrt2}$ and a vector with $\alpha = \tfrac{1}{\sqrt2}$, $\beta = -\tfrac{1}{\sqrt2}$ give *identical* measurement probabilities ($\tfrac12$ each), so no "hidden coin" story can tell them apart — and yet they are different vectors, and section 5 of this notebook builds a circuit that reliably outputs `0` for one and `1` for the other. The vector is the thing. Hold on to the vector.

Let's write one down in plain NumPy first, with no simulator involved.

In [ ]:
import numpy as np

# A perfectly ordinary unit vector in C^2. Note the second entry is imaginary:
# amplitudes are complex numbers, and nothing here forbids that.
psi = np.array([0.6, 0.8j])

# np.abs of a complex array gives the modulus |z| = sqrt(re^2 + im^2) elementwise,
# so np.abs(psi) ** 2 is the vector of |amplitude|^2 — the two probabilities.
probabilities = np.abs(psi) ** 2

print("amplitudes:   ", psi)
print("probabilities:", probabilities)
print("they sum to:  ", probabilities.sum())

So this qubit reads `0` with probability $0.36$ and `1` with probability $0.64$.

Notice what got thrown away in the second line. The amplitude $0.8i$ and the amplitude $0.8$ and the amplitude $-0.8$ all have modulus $0.8$, so all three give the same probability $0.64$. The *direction* of the complex number — its **phase** — leaves no trace in the probabilities at all.

You are entitled to be suspicious of that. If the phase never shows up in any measurement, why carry it? Why is $\mathbb{C}^2$ the right space instead of $\mathbb{R}^2$, or instead of just a probability distribution over $\{0,1\}$?

Section 5 answers that, and the answer is the reason this notebook exists. Until then, hold the question open.

## 2. Dirac notation, from scratch

Physicists do not write $\begin{pmatrix}\alpha\\\beta\end{pmatrix}$. They write $\alpha|0\rangle + \beta|1\rangle$. This is **Dirac notation**, and it is worth ten minutes because every quantum text and every later notebook uses it without apology.

### The ket

A **ket**, written $|\text{something}\rangle$, is a column vector. The label inside the bars is just a *name* — it carries no arithmetic meaning, it only says which vector we mean. The two standard basis vectors of $\mathbb{C}^2$ get the names `0` and `1`:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}.$$

These two together are called the **computational basis**: the basis in which "0" and "1" mean what a classical computer means by them. Any qubit state is a linear combination of them,

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle,$$

which is exactly the column vector from before, written in the basis. A linear combination of basis states like this is what physics calls a **superposition**. The word means precisely "linear combination" and nothing more mysterious than that.

### The bra

A **bra**, written $\langle\psi|$, is the conjugate transpose of the ket $|\psi\rangle$ — a *row* vector, with every entry complex-conjugated:

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta\end{pmatrix} \quad \Longrightarrow \quad \langle\psi| = \begin{pmatrix} \bar\alpha & \bar\beta\end{pmatrix}.$$

Bras exist so that the inner product can be written as a row meeting a column:

$$\langle\phi|\psi\rangle = \begin{pmatrix}\bar\gamma & \bar\delta\end{pmatrix}\begin{pmatrix}\alpha\\\beta\end{pmatrix} = \bar\gamma\alpha + \bar\delta\beta.$$

That is the standard Hermitian inner product you already know. In particular $\langle\psi|\psi\rangle = |\alpha|^2 + |\beta|^2 = 1$: "the state is normalized" is "the bra meets its own ket and gives 1".

### Why the notation earns its keep

Three reasons, all of which will bite you later if you skip them:

- **The label survives.** $|0101\rangle$ names a basis state of a *four*-qubit system without you having to work out that it is component 5 of a 16-component vector. When we get to Shor's algorithm, $|x\rangle$ will name "the register holding the number $x$", and the notation keeps the meaning attached to the vector.
- **The bracket is unambiguous.** $\langle\phi|\psi\rangle$ is a number (an inner product). $|\psi\rangle\langle\phi|$ is a matrix (an outer product). The shape of the expression tells you the shape of the result.
- **The conjugation is explicit.** The moment amplitudes are complex, the difference between $|\psi\rangle$ and $\langle\psi|$ is a real difference, and the notation refuses to let you forget it.

Here is the whole thing in NumPy.

In [ ]:
ket_0 = np.array([1, 0], dtype=complex)   # |0>
ket_1 = np.array([0, 1], dtype=complex)   # |1>

alpha, beta = 0.6, 0.8j
ket_psi = alpha * ket_0 + beta * ket_1    # |psi> = 0.6|0> + 0.8i|1>

# The bra is the conjugate transpose. For a 1-D NumPy array "transpose" is a no-op,
# so all that is left of the conjugate transpose is .conj().
bra_psi = ket_psi.conj()

print("|psi> =", ket_psi)
print("<psi| =", bra_psi)

# np.vdot(u, v) computes sum(conj(u_i) * v_i) — it conjugates its FIRST argument.
# That is exactly the bra-meets-ket operation <psi|psi>, so we do not conjugate
# by hand here. (np.dot would NOT conjugate, and would give the wrong answer.)
print("<psi|psi> =", np.vdot(ket_psi, ket_psi))

$\langle\psi|\psi\rangle = 1$, as required — and it came out as a real number even though the amplitudes were complex, because conjugating one side turns $\beta\beta$ into $\bar\beta\beta = |\beta|^2$. That is *why* the inner product conjugates: without it, a normalized vector would not have length 1.

Compare: `np.dot(ket_psi, ket_psi)` would give $0.36 + (0.8i)^2 = 0.36 - 0.64 = -0.28$. Not a length. Whenever you take an inner product of complex vectors in NumPy, reach for `np.vdot`.

### qsim speaks Dirac notation

The simulator will print any state this way. Below we build a state with a genuinely complex amplitude — apply `H` and then `T` to a fresh qubit — and print both its ket and its bra.

Don't worry about the three lines that build the circuit; section 3 takes them apart properly. For now, just watch the two outputs.

In [ ]:
from qsim import Circuit, viz
from qsim.gates import (
    CNOT,
    SWAP,
    H,
    Rx,
    Ry,
    Rz,
    T,
    X,
    Z,
)

qc = Circuit(name="a complex amplitude", seed=1234)
q = qc.alloc()
H(q)   # put the qubit in an equal superposition
T(q)   # rotate the |1> amplitude's phase by 45 degrees

qc.inspect.ket()

In [ ]:
qc.inspect.bra()

Read those two lines side by side:

$$|\psi\rangle = 0.707\,|0\rangle + (0.500 + 0.500i)\,|1\rangle \qquad\text{versus}\qquad \langle\psi| = 0.707\,\langle 0| + (0.500 - 0.500i)\,\langle 1|.$$

The only difference is the sign of the imaginary part: `+0.500i` became `-0.500i`. That is the conjugation, made visible. Nothing else changed — a bra carries no information its ket doesn't; it is the same state written so it can sit on the left of an inner product.

(In Jupyter, `ket()` and `bra()` render as typeset mathematics rather than plain text. If you are reading executed output, that is what you are seeing.)

## 3. Your first circuit

Three objects do all the work in `qsim`:

- **`Circuit`** owns the state. There is exactly one state vector, and the circuit holds it. Pass `seed=` to make measurement outcomes reproducible — the randomness of quantum measurement is real, but a notebook that gives different numbers every run is hard to write prose about.
- **A qubit handle**, returned by `qc.alloc()` (one qubit) or `qc.alloc_many(n)` (a tuple of $n$). Two functions rather than one that returns a qubit *or* a tuple depending on its argument, so you always know what you got. A handle is a *name for an axis of the circuit's state* — it does not carry a state around with it. That distinction looks pedantic now and becomes the whole story in notebook 02.
- **Gates**, imported from `qsim.gates`. A gate is a callable that mutates the circuit and returns `None`: you write `H(a)`, not `a = H(a)`. It finds its circuit from the handle you passed it.

The gate we start with is **H**, the *Hadamard* gate. As a matrix,

$$H = \frac{1}{\sqrt2}\begin{pmatrix} 1 & 1 \\ 1 & -1\end{pmatrix}, \qquad\text{so}\qquad H|0\rangle = \frac{1}{\sqrt2}\begin{pmatrix}1\\1\end{pmatrix} = \frac{|0\rangle + |1\rangle}{\sqrt2}.$$

Applying a gate is matrix-vector multiplication, and that is *all* it is. The state $\frac{|0\rangle+|1\rangle}{\sqrt2}$ comes up so often it has its own name, $|{+}\rangle$.

In [ ]:
qc = Circuit(name="one qubit", seed=1234)
a = qc.alloc()          # a fresh qubit always starts in |0>

print("before H:", qc.inspect.ket())
H(a)
print("after H: ", qc.inspect.ket())

# state_vector() flattens the state to a plain NumPy array of complex amplitudes,
# one per basis state, in the order |0>, |1>.
print("raw amplitudes:", qc.inspect.state_vector())
print("norm:          ", qc.inspect.norm())

$0.707 \approx 1/\sqrt2$, so the state is $\frac{1}{\sqrt2}|0\rangle + \frac{1}{\sqrt2}|1\rangle$, exactly as the matrix multiplication predicted.

`norm()` prints `0.9999999999999999` rather than `1.0`. That is floating point, not physics — $1/\sqrt2$ is not exactly representable in binary. Every gate is a **unitary** matrix, meaning it preserves the length of the vector it multiplies ($U^\dagger U = I$), so this number must stay at 1 forever. It is a useful running sanity check: if it ever drifts, something is broken. "Unitary" is worth remembering as a word, because it is the precise sense in which a gate is "a thing that can physically happen": lengths are total probability, and total probability cannot change.

Now let's look at the state instead of reading it.

In [ ]:
# Every viz function returns the matplotlib Figure it drew, so you can keep
# tweaking it. Assigning it (rather than leaving it as the cell's last
# expression) is what stops Jupyter from drawing the same figure twice.
fig = viz.amplitudes(qc)

Two bars of equal height $0.707$. The height of each bar is $|\text{amplitude}|$; the **color** is the amplitude's phase, read off the color wheel in the legend on the right. Both amplitudes here are positive real numbers, i.e. phase $0$, so both bars are the same color. Remember that the colors are there. In section 4 they start doing something.

### Where the probabilities come from: the Born rule

The rule connecting the vector to what you actually observe has a name — the **Born rule** — and you have already seen it:

$$\Pr[\text{outcome } b] = |\langle b|\psi\rangle|^2 = |\text{amplitude of } |b\rangle|^2.$$

It is a postulate. It is not derived from the rest of the formalism; it is the bridge between the vector and the laboratory, and it is where all the randomness in quantum mechanics enters. Everything else — every gate, every circuit in every later notebook — is deterministic matrix multiplication.

`qsim` gives you the rule two ways, and the difference between them matters:

- `qc.inspect.probabilities()` returns the exact numbers $|\alpha|^2, |\beta|^2$. **No real machine can do this.** You cannot read amplitudes off a quantum computer; that is why this lives behind the `inspect` namespace, whose whole job is to hold the operations that are physically impossible. Everything inside `inspect` is cheating, and it is labelled as such so you always know when you are cheating.
- `qc.inspect.sample(1000)` simulates running the circuit 1000 times and counting outcomes. This is what an experiment gives you: a tally, with statistical noise, that only approximates the exact numbers. (The cheat here is subtler — a real machine would have to rerun the whole circuit for each shot, because the first measurement destroys the state. `sample` draws from the intact state and leaves it intact.)

In [ ]:
print("exact probabilities:", qc.inspect.probabilities())
print("1000 simulated runs:", qc.inspect.sample(1000))

The exact answer is `[0.5, 0.5]`; the tally lands near 500/500 without hitting it. That gap is not simulator error, it is the same $\sqrt{N}$ statistical noise you would fight in a real experiment: to know a probability to three decimal places you need on the order of a million shots.

One more display: putting the circuit itself as the last expression in a cell renders a table of its largest amplitudes — bar length for magnitude, color for phase, and the numeric value of each amplitude on the right.

In [ ]:
qc

### Every gate has two names

The one-letter names are what you will see in every textbook and on every circuit
diagram, but they are opaque until you have memorized them. So each gate in qsim is
also importable under a spelled-out name, and the two are *the same object* —
`Hadamard is H` is `True`. Use whichever reads better on a given line.

| Symbol | Full name | What it does |
|---|---|---|
| `H` | `Hadamard` | sends $\|0\rangle$ to the equal superposition |
| `X` | `PauliX` | flips the bit: $\|0\rangle \leftrightarrow \|1\rangle$ |
| `Y` | `PauliY` | flips the bit *and* the phase |
| `Z` | `PauliZ` | flips the phase: negates the $\|1\rangle$ amplitude |
| `S` | `SqrtZ` | quarter turn about $z$; applied twice it is $Z$ |
| `T` | `FourthRootZ` | eighth turn about $z$; applied four times it is $Z$ |
| `SX` | `SqrtX` | half a bit flip; applied twice it is $X$ |
| `SWAP` | `Swap` | exchanges two qubits |
| `CNOT` | `ControlledNot` | flips the target if the control is 1 |
| `CZ` | `ControlledZ` | negates only the $\|11\rangle$ amplitude |
| `Toffoli` | `ControlledControlledNot` | flips the target if *both* controls are 1 |
| `Fredkin` | `ControlledSwap` | swaps two targets if the control is 1 |
| `Rx`, `Ry`, `Rz` | `RotationX/Y/Z` | rotate by `theta` about that axis |
| `Phase` | — | multiplies the $\|1\rangle$ amplitude by $e^{i\theta}$ |
| `CPhase` | `ControlledPhase` | the same, but only on $\|11\rangle$ |

$S$ and $T$ are the two with no settled name in the literature, so qsim names them
for what they do. You will meet most of these properly later; the table is here to
be come back to.

Whichever name you type, the *short* one is what gets recorded in the circuit's
history and printed by `gate_counts()`, so diagrams stay compact.

## 4. Phases are invisible...

Now we take the question from section 1 seriously. Meet **Z**, the phase-flip gate:

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1\end{pmatrix}, \qquad Z|0\rangle = |0\rangle, \qquad Z|1\rangle = -|1\rangle.$$

$Z$ leaves the $|0\rangle$ amplitude alone and negates the $|1\rangle$ amplitude. Applied to $|{+}\rangle$ it produces the other famous state:

$$Z\,|{+}\rangle = Z\,\frac{|0\rangle + |1\rangle}{\sqrt2} = \frac{|0\rangle - |1\rangle}{\sqrt2} = |{-}\rangle.$$

$|{+}\rangle$ and $|{-}\rangle$ are genuinely different vectors — they are orthogonal, in fact: $\langle{+}|{-}\rangle = \frac{1}{2}(1 \cdot 1 + 1 \cdot (-1)) = 0$. As different as $|0\rangle$ and $|1\rangle$ are.

And yet.

In [ ]:
plus = Circuit(name="|+>", seed=1)
p = plus.alloc()
H(p)

minus = Circuit(name="|->", seed=1)
m = minus.alloc()
H(m)
Z(m)                      # the only difference between the two circuits

print("|+> =", plus.inspect.ket(), "   probabilities:", plus.inspect.probabilities())
print("|-> =", minus.inspect.ket(), "   probabilities:", minus.inspect.probabilities())

Two orthogonal states — the most distinguishable a pair of states can possibly be — and **identical measurement statistics**: 50/50 for both. Sample either of them a billion times and you could not tell which one you had.

This is not a limitation of our simulator or of current hardware. It is the Born rule doing what it says: probabilities are squared magnitudes, and $|-0.707|^2 = |0.707|^2$. Measuring in the computational basis simply cannot see the minus sign.

Look at the two pictures.

In [ ]:
fig = viz.amplitudes(plus)

In [ ]:
fig = viz.amplitudes(minus)

Compare the two figures carefully.

- **The bar heights are identical.** Both plots: two bars at $0.707$. This is the same fact as the probability lists above — heights are $|\text{amplitude}|$, and measurement only ever sees $|\text{amplitude}|^2$.
- **The colors are not.** In the $|{+}\rangle$ plot both bars are the same hue (red, phase $0$). In the $|{-}\rangle$ plot the $|0\rangle$ bar is still red, and the $|1\rangle$ bar has turned **cyan** — the color halfway around the wheel, phase $\pi$. That is what "$\times(-1)$" looks like on the color scale: $-1 = e^{i\pi}$.

So the state carries information that no measurement can read. There are only two honest reactions to that, and one of them is wrong:

1. *The phase is bookkeeping — an artifact of a redundant description.* Reasonable! And false.
2. *The phase is real, and there must be a way to make it matter.* Correct. Read on.

## 5. ...until they interfere

This is the section the whole notebook is built around. Take it slowly.

The trick is that we do not have to measure the state while the phase is hidden. We can apply *another gate first* — one that mixes the two amplitudes together so that the invisible sign decides whether they add or cancel.

The gate that does the mixing is $H$ again. Note what $H$ does to *both* basis states:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt2}, \qquad H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt2}.$$

Each input basis state fans out into *both* output basis states. So if the input is a superposition, each output amplitude gets a contribution from **two different paths** — one through $|0\rangle$ and one through $|1\rangle$ — and those two contributions are complex numbers that get **added**. Adding complex numbers can make them bigger, or smaller, or exactly zero. That is the whole mechanism.

Let's do both computations by hand before running anything.

### Path A: $H\,H\,|0\rangle$

$$H|0\rangle = \tfrac{1}{\sqrt2}|0\rangle + \tfrac{1}{\sqrt2}|1\rangle.$$

Now apply $H$ again, using linearity — $H$ acts on each term separately:

$$H\left(\tfrac{1}{\sqrt2}|0\rangle + \tfrac{1}{\sqrt2}|1\rangle\right) = \tfrac{1}{\sqrt2}\cdot\frac{|0\rangle + |1\rangle}{\sqrt2} \;+\; \tfrac{1}{\sqrt2}\cdot\frac{|0\rangle - |1\rangle}{\sqrt2}.$$

Collect the $|0\rangle$ terms and the $|1\rangle$ terms:

$$= \underbrace{\left(\tfrac12 + \tfrac12\right)}_{\text{two paths, same sign}}|0\rangle \;+\; \underbrace{\left(\tfrac12 - \tfrac12\right)}_{\text{two paths, opposite signs}}|1\rangle \;=\; 1\cdot|0\rangle + 0\cdot|1\rangle \;=\; |0\rangle.$$

The $|1\rangle$ amplitude is **exactly zero**. Two ways to arrive at the outcome `1`, and they cancel. Measuring now gives `0` with probability 1.

### Path B: $H\,Z\,H\,|0\rangle$

Identical, except the middle $Z$ flips the sign of the $|1\rangle$ amplitude before the second $H$:

$$Z\,H|0\rangle = \tfrac{1}{\sqrt2}|0\rangle - \tfrac{1}{\sqrt2}|1\rangle.$$

$$H\left(\tfrac{1}{\sqrt2}|0\rangle - \tfrac{1}{\sqrt2}|1\rangle\right) = \tfrac{1}{\sqrt2}\cdot\frac{|0\rangle + |1\rangle}{\sqrt2} \;-\; \tfrac{1}{\sqrt2}\cdot\frac{|0\rangle - |1\rangle}{\sqrt2}$$

$$= \left(\tfrac12 - \tfrac12\right)|0\rangle + \left(\tfrac12 + \tfrac12\right)|1\rangle = 0\cdot|0\rangle + 1\cdot|1\rangle = |1\rangle.$$

**The cancellation moved.** One sign change in the middle of the circuit, invisible to any measurement taken at that moment, and the guaranteed output flipped from `0` to `1`.

The two names for the two cases: amplitudes reinforcing is **constructive interference**, amplitudes cancelling is **destructive interference**. This is the same interference as two water waves meeting, or the two slits in Young's experiment, and it is arithmetic on the amplitudes, not a metaphor.

Now run it.

In [ ]:
without_z = Circuit(name="H H |0>", seed=1)
w = without_z.alloc()
H(w)
H(w)

with_z = Circuit(name="H Z H |0>", seed=1)
v = with_z.alloc()
H(v)
Z(v)
H(v)

print("H H   |0> =", without_z.inspect.ket(), "  probabilities:", without_z.inspect.probabilities())
print("H Z H |0> =", with_z.inspect.ket(), "  probabilities:", with_z.inspect.probabilities())

`[1. 0.]` and `[0. 1.]`. Not "roughly", not "with high probability" — **certainty**, both times, and opposite certainties.

Sit with the sequence of events for a moment, because it is genuinely strange:

| after | state | what a measurement would give |
|---|---|---|
| $H$ | $\frac{|0\rangle+|1\rangle}{\sqrt2}$ | 50/50 |
| $H$, then $Z$ | $\frac{|0\rangle-|1\rangle}{\sqrt2}$ | 50/50 — *indistinguishable from the row above* |
| $H, H$ | $|0\rangle$ | always 0 |
| $H, Z, H$ | $|1\rangle$ | always 1 |

Halfway through, the two circuits are in states no experiment can tell apart. At the end, they are in states no experiment can *confuse*. The information that made the difference was in the relative phase the entire time — carried, invisible, until the second $H$ converted it into something measurable.

That is the answer to section 1's question. **The amplitudes are complex because we need a quantity that can cancel.** Probabilities are non-negative; sums of non-negative numbers only ever get bigger. If a qubit really were a coin whose value we merely didn't know, the second $H$ would have to *randomize further*, never sharpen to certainty. Every quantum algorithm in the later notebooks — Grover's search, phase estimation, Shor's factoring — is the same idea scaled up: arrange a superposition so that the amplitudes of all the wrong answers destructively interfere, and the right answer is what's left.

### The knob between the two cases

$Z$ is an all-or-nothing sign flip. `Rz(q, theta=...)` is the continuous version: it rotates the *relative phase* between the two amplitudes by an angle $\theta$ of your choosing, with $\theta = \pi$ reproducing $Z$. (Angles in `qsim` are keyword-only — every positional argument to a gate is a qubit, so a bare number in that slot would be the one thing a reader has to stop and decode.)

Putting `Rz(theta)` between two Hadamards and repeating the hand computation gives

$$H\,R_z(\theta)\,H\,|0\rangle = \cos(\theta/2)\,|0\rangle - i\sin(\theta/2)\,|1\rangle, \qquad \Pr[1] = \sin^2(\theta/2),$$

which sweeps smoothly from full cancellation at $\theta = 0$ to full reinforcement at $\theta = \pi$. Let's watch it.

In [ ]:
import matplotlib.pyplot as plt

# np.linspace(0, 2*pi, 61) gives 61 evenly spaced angles covering a full turn.
thetas = np.linspace(0, 2 * np.pi, 61)
p_one = []

for theta in thetas:
    qc_sweep = Circuit()
    s = qc_sweep.alloc()
    H(s)
    Rz(s, theta=theta)     # rotate the relative phase by theta
    H(s)                   # ... and convert phase into probability
    # probabilities() is indexed by basis state, so entry 1 is Pr[measure 1].
    p_one.append(qc_sweep.inspect.probabilities()[1])

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(thetas, p_one, linewidth=2, label=r"measured: $\Pr[1]$")
ax.plot(thetas, np.sin(thetas / 2) ** 2, "k--", linewidth=1, label=r"predicted: $\sin^2(\theta/2)$")
ax.set_xlabel(r"phase $\theta$ applied between the two Hadamards")
ax.set_ylabel(r"$\Pr[\text{measure } 1]$")
ax.set_title("an invisible phase, converted into a probability")
ax.legend()

The simulated points sit exactly on the predicted curve. Read the two ends of it:

- At $\theta = 0$ there is no phase to convert, the two paths to `1` cancel perfectly, and $\Pr[1] = 0$.
- At $\theta = \pi$ (which is $Z$) the cancellation has moved to the `0` outcome and $\Pr[1] = 1$.

Everything in between is a partial cancellation. **This curve is an interference fringe** — the same shape you would measure in a double-slit experiment by moving the detector, produced here by turning a knob that no measurement, taken at the time you turn it, could detect.

## 6. The Bloch sphere

We have been drawing states as bar charts. For a *single* qubit there is a better picture, and it is worth building carefully because every later notebook uses it.

Start by counting the real parameters in $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$. Two complex numbers is **4** real numbers. Then:

- Normalization, $|\alpha|^2 + |\beta|^2 = 1$, is one equation: **3** left.
- **Global phase is unobservable.** Multiplying the *entire* state by $e^{i\varphi}$ changes no measurement result anywhere, ever — every probability is $|e^{i\varphi}\gamma|^2 = |\gamma|^2$, and the factor survives every gate untouched, so no later circuit can catch it either. (Be careful: this is *global* phase, one factor multiplying everything. The **relative** phase between $\alpha$ and $\beta$ — the thing section 5 was about — is emphatically observable.) So $|\psi\rangle$ and $e^{i\varphi}|\psi\rangle$ are the same physical state, and we may use that freedom to make $\alpha$ real and non-negative: **2** left.

Two real parameters, with a normalization constraint — that is a sphere. Every single-qubit state is a point on the unit sphere in $\mathbb{R}^3$, called the **Bloch sphere**. The dictionary:

| direction | state |
|---|---|
| $+z$ (north pole) | $|0\rangle$ |
| $-z$ (south pole) | $|1\rangle$ |
| $+x$ | $|{+}\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}$ |
| $-x$ | $|{-}\rangle = \frac{|0\rangle-|1\rangle}{\sqrt2}$ |
| $+y$ | $\frac{|0\rangle+i|1\rangle}{\sqrt2}$ |

Note the first surprise: $|0\rangle$ and $|1\rangle$ are *orthogonal* as vectors in $\mathbb{C}^2$, but they sit at *opposite poles* — 180° apart — on the sphere. Orthogonal states are antipodal points, not perpendicular directions. The sphere is a picture of the state space, not of the vector space.

`qc.inspect.bloch_vector(q)` returns the $(x, y, z)$ coordinates. Its components are the average values of the three Pauli measurements $X$, $Y$, $Z$ — each is "how strongly does the state point along this axis". Let's map out the states we've built.

In [ ]:
labels_and_builders = [
    ("|0>",   lambda c, q: None),
    ("|1>",   lambda c, q: X(q)),
    ("|+>",   lambda c, q: H(q)),
    ("|->",   lambda c, q: (H(q), Z(q))),
    ("T|+>",  lambda c, q: (H(q), T(q))),
]

for label, build in labels_and_builders:
    c = Circuit()
    q = c.alloc()
    build(c, q)
    x, y, z = c.inspect.bloch_vector(q)
    print(f"{label:>6}  ->  x={x:+.3f}  y={y:+.3f}  z={z:+.3f}")

Exactly the dictionary above: $|0\rangle$ at the north pole $(0,0,1)$, $|1\rangle$ at the south pole, $|{+}\rangle$ and $|{-}\rangle$ at opposite ends of the $x$-axis.

The interesting row is the last one. `T` multiplies the $|1\rangle$ amplitude by $e^{i\pi/4}$, and $T|{+}\rangle$ landed at $(0.707, 0.707, 0)$ — still on the equator, rotated **45° around the $z$-axis** from $|{+}\rangle$. That is the general pattern:

> **Every single-qubit gate is a rotation of the Bloch sphere.**

`Rz(theta)` rotates by $\theta$ about the $z$-axis; `Rx` and `Ry` about the other two. $Z$ is a half-turn about $z$ (which is why it maps $|{+}\rangle$ to $|{-}\rangle$ — antipodal points on the equator), $X$ is a half-turn about $x$ (which is why it swaps the poles), and $H$ is a half-turn about the diagonal axis $(x+z)/\sqrt2$, which exchanges the $x$ and $z$ axes — that is *precisely* why $H$ converts a phase difference (an angle in the $xy$-plane, invisible to a $z$-measurement) into a $z$-position (visible). Section 5, in one sentence of geometry.

Here is the sphere itself.

In [ ]:
sphere_0 = Circuit(name="|0>", seed=1)
s0 = sphere_0.alloc()
fig = viz.bloch(sphere_0, s0)

In [ ]:
sphere_plus = Circuit(name="|+>", seed=1)
sp = sphere_plus.alloc()
H(sp)
fig = viz.bloch(sphere_plus, sp)

In [ ]:
sphere_t = Circuit(name="T|+>", seed=1)
st = sphere_t.alloc()
H(st)
T(st)
fig = viz.bloch(sphere_t, st)

$H$ tipped the arrow from the pole down to the equator; $T$ then walked it 45° around. The plot titles report the arrow's **length**, which is $1.00$ in all three cases.

That number is not decoration. Length exactly 1 means the qubit is in a **pure state** — it has a state of its own, a definite vector in $\mathbb{C}^2$. Arrows *shorter* than 1 are possible, they live strictly inside the ball, and they describe qubits that do not have a state of their own. Notebook 02 produces a length-$0$ arrow — an arrow that is just a dot at the center — and that turns out to be the signature of entanglement. Watch for it.

A last caution about the picture: the Bloch sphere is a single-qubit luxury. Two qubits do not have a 6-dimensional version of it; the whole point of the next notebook is that the joint state is not a pair of separate arrows.

## 7. Measurement collapses the state

Every gate so far has been a unitary matrix: reversible, deterministic, length-preserving. **Measurement is the exception**, and it is the only one.

`qc.measure(q)` does two things:

1. Returns `0` or `1`, chosen at random with the Born-rule probabilities.
2. **Changes the state** so it agrees with what was reported. If the outcome was `0`, the amplitude of $|1\rangle$ is set to zero and what remains is rescaled back to unit length. The state is now $|0\rangle$, full stop.

Step 2 is called **collapse**, and it is where information is destroyed — irreversibly. The superposition that was there before is gone; no subsequent gate can recover it. This is the sharpest difference between `qc.measure()` (real: collapses) and `qc.inspect.sample()` (a cheat: peeks at the distribution and leaves the state alone).

Let's watch it happen. Each row below is a *fresh* circuit with a different seed, prepared in $|{+}\rangle$ and then measured.

In [ ]:
# A master generator hands out the per-circuit seeds, so the runs are independent.
master = np.random.default_rng(0)

for trial in range(6):
    qc_m = Circuit(seed=int(master.integers(0, 2**31)))
    q_m = qc_m.alloc()
    H(q_m)
    before = qc_m.inspect.ket()
    outcome = qc_m.measure(q_m)
    print(f"before: {str(before):<28} measured: {outcome}   after: {qc_m.inspect.ket()}")

Read the columns. Every row starts from the *same* state, $0.707|0\rangle + 0.707|1\rangle$. The middle column differs from row to row — that is genuine randomness, the only randomness in the whole formalism. And the right column is never a superposition: it is $1.000|0\rangle$ or $1.000|1\rangle$, matching the outcome.

The state did not "reveal" the answer. It *became* the answer.

Which has a consequence you can check: measure the same qubit twice and the second measurement is not random any more, because the first one already collapsed the state it would have sampled from.

In [ ]:
qc_twice = Circuit(seed=99)
q_twice = qc_twice.alloc()
H(q_twice)

first = qc_twice.measure(q_twice)
print("first measurement: ", first, " state now:", qc_twice.inspect.ket())

for repeat in range(5):
    print("  measured again:  ", qc_twice.measure(q_twice))

The same answer, every time, forever. A repeated measurement of an unchanged qubit is not an experiment — it is a lookup of a fact already established. (This is exactly the behaviour you want from a classical bit, and it is worth noticing that *measurement is what makes qubits behave classically*. That thought is the seed of notebook 05, where classical behaviour emerges from a quantum world rather than being bolted onto it.)

Finally: individually random, collectively lawful. Two hundred independent circuits, each prepared in $|{+}\rangle$ and measured once.

In [ ]:
from collections import Counter

tally = Counter()
master = np.random.default_rng(0)
for trial in range(200):
    qc_t = Circuit(seed=int(master.integers(0, 2**31)))
    q_t = qc_t.alloc()
    H(q_t)
    tally[qc_t.measure(q_t)] += 1

print("200 fresh circuits, one measurement each:", dict(tally))

About 100 each, give or take the $\pm\sqrt{200}/2 \approx 7$ you would expect from a fair coin. The Born rule is a statement about ensembles: it predicts the histogram exactly and the individual outcome not at all.

## 8. Two qubits, and the tensor product

One qubit lives in $\mathbb{C}^2$. Two qubits live in $\mathbb{C}^2 \otimes \mathbb{C}^2 = \mathbb{C}^4$: four basis states, one per bit pattern, named $|00\rangle, |01\rangle, |10\rangle, |11\rangle$. Dimension **multiplies** where classical state counts also multiply — $n$ qubits need $2^n$ amplitudes, which is why simulating 50 qubits is out of reach and why this notebook stays small.

If two qubits are prepared independently, in states $|\phi\rangle = \begin{pmatrix}a_0\\a_1\end{pmatrix}$ and $|\chi\rangle = \begin{pmatrix}b_0\\b_1\end{pmatrix}$, the joint state is the **tensor product** $|\phi\rangle \otimes |\chi\rangle$, whose amplitudes are all the products:

$$\text{amplitude of } |ij\rangle \;=\; a_i \, b_j.$$

Here is the design decision at the heart of `qsim`. We could store those four numbers as a flat vector of length 4. Instead the state is a NumPy array of **shape `(2, 2)`** — and for $n$ qubits, shape `(2,) * n`. The reason is that then *the axes of the array are the qubits*: axis 0 is qubit 0, axis 1 is qubit 1, and

$$\texttt{psi[i, j]} \;=\; \text{the amplitude of } |ij\rangle.$$

Indexing the array with a bit pattern gives you the amplitude of that bit pattern. No index arithmetic, no $2i + j$ anywhere. A gate on one qubit becomes a $2\times2$ matrix applied along one axis. This identification — *array axes are tensor factors* — is the single most important idea in this codebase.

One convention, stated once and obeyed everywhere: **qubit 0 is the most significant bit.** So $|10\rangle$ means qubit 0 is 1 and qubit 1 is 0, and read as a binary integer it is 2, not 1.

In [ ]:
two = Circuit(name="two qubits", seed=5)
a, b = two.alloc_many(2)     # a tuple of two handles; a is qubit 0, b is qubit 1
X(a)                         # flip qubit 0, leaving qubit 1 in |0>

tensor = two.inspect.state_tensor()
print("shape:", tensor.shape)      # (2, 2) -- one axis per qubit
print(tensor.real)                 # .real only to keep the printout readable
print()
print("state:              ", two.inspect.ket())
print("psi[1, 0]:          ", tensor[1, 0])
print("inspect.amplitude:  ", two.inspect.amplitude('10'))

The single `1` sits at index `[1, 0]` — qubit 0 is 1, qubit 1 is 0 — and both the printed ket and `amplitude("10")` agree. The bit pattern *is* the index tuple.

Now the bit-order convention, checked rather than asserted. `qc.register(n)` allocates $n$ qubits as an ordered group, and `measure_all` measures the whole group and returns the result as one integer with `reg[0]` as the most significant bit.

In [ ]:
msb = Circuit(name="MSB check", seed=5)
reg = msb.register(2, name="r")
X(reg[0])                    # set the FIRST qubit of the register to |1>

print("state:", msb.inspect.ket())
print("measure_all ->", msb.measure_all(reg), " (2, not 1: reg[0] is the most significant bit)")

### Four basis states at once

Apply $H$ to both qubits of $|00\rangle$. Each $H$ produces a two-term superposition, and the tensor product multiplies them out:

$$H|0\rangle \otimes H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2} \otimes \frac{|0\rangle+|1\rangle}{\sqrt2} = \tfrac12\big(|00\rangle + |01\rangle + |10\rangle + |11\rangle\big).$$

Four equal amplitudes of $\tfrac12$ — and $4 \times (\tfrac12)^2 = 1$, as it must be. With $n$ Hadamards you get all $2^n$ bit patterns at once with equal amplitude, which is the opening move of nearly every quantum algorithm.

In [ ]:
four = Circuit(name="H on both", seed=5)
c0, c1 = four.alloc_many(2)
H(c0)
H(c1)

print(four.inspect.ket())
fig = viz.probabilities(four, top=8)

In [ ]:
fig = viz.amplitudes(four)

### Product states factor — and the tensor is the outer product

If the two qubits were prepared independently, the joint amplitude array is literally the outer product of the two single-qubit vectors, $\texttt{psi[i, j]} = a_i b_j$. Let's build such a state with two different rotations and check that claim against NumPy.

`Ry(q, theta)` rotates about the $y$-axis of the Bloch sphere and takes $|0\rangle$ to $\cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$ — real amplitudes, easy to write down by hand. (The half-angles are not a typo: a full $2\pi$ turn of the Bloch vector multiplies the state vector by $-1$, so state vectors pick up only half the rotation angle. Spin-1/2 systems really do behave this way.)

In [ ]:
prod = Circuit(name="a product state", seed=5)
u, v = prod.alloc_many(2)
Ry(u, theta=1.0)
Ry(v, theta=2.0)

# What each qubit is on its own, worked out by hand from the Ry formula:
vec_u = np.array([np.cos(0.5), np.sin(0.5)])   # theta/2 = 0.5
vec_v = np.array([np.cos(1.0), np.sin(1.0)])   # theta/2 = 1.0

# np.multiply.outer(u, v)[i, j] == u[i] * v[j] — every product of one entry from
# each vector, arranged into a (2, 2) array. That is exactly the tensor product.
predicted = np.multiply.outer(vec_u, vec_v)

print("simulator's tensor:\n", prod.inspect.state_tensor().real.round(4))
print("\nouter product:\n", predicted.round(4))
# np.allclose compares elementwise within a small tolerance, since these are floats.
matches = np.allclose(prod.inspect.state_tensor(), predicted)
print("\nequal to within floating-point error:", matches)

They match. Every state built by acting on the two qubits *separately* factors this way.

Which raises the obvious question — **can every two-qubit state be written as an outer product?** Four amplitudes, and only $2+2$ parameters on the right-hand side to fit them with. The counting says no, and that gap is the subject of notebook 02.

### Two-qubit gates: SWAP is three CNOTs

A **CNOT** ("controlled-NOT") takes two qubits, a control and a target: it applies $X$ to the target if the control is $|1\rangle$, and does nothing if the control is $|0\rangle$. By linearity, on a superposed control it does *both at once* — which is how entanglement gets made, next notebook.

$$\text{CNOT}\,|00\rangle = |00\rangle, \quad \text{CNOT}\,|01\rangle = |01\rangle, \quad \text{CNOT}\,|10\rangle = |11\rangle, \quad \text{CNOT}\,|11\rangle = |10\rangle.$$

**SWAP** exchanges the two qubits outright. The classic identity is that you do not need SWAP as a primitive — three alternating CNOTs do the job:

$$\text{SWAP}(a,b) = \text{CNOT}(a,b)\,\text{CNOT}(b,a)\,\text{CNOT}(a,b).$$

You can verify this by hand on the four basis states (try $|10\rangle$: it becomes $|11\rangle$, then $|01\rangle$, then $|01\rangle$ — swapped), and linearity extends it to everything else. But let's check it on a state chosen to be as un-symmetric as possible, with complex amplitudes and no accidental structure.

In [ ]:
def prepare(name):
    # A deliberately lopsided two-qubit state, with a complex amplitude in it,
    # so that the comparison below cannot succeed by accident.
    c = Circuit(name=name, seed=11)
    x, y = c.alloc_many(2)
    Ry(x, theta=0.7)
    Rx(y, theta=1.1)
    T(x)                      # T makes one amplitude genuinely complex
    return c, x, y

by_swap, x1, y1 = prepare("SWAP")
SWAP(x1, y1)

by_cnots, x2, y2 = prepare("three CNOTs")
CNOT(x2, y2)
CNOT(y2, x2)
CNOT(x2, y2)

print("SWAP:       ", by_swap.inspect.ket())
print("three CNOTs:", by_cnots.inspect.ket())

# np.max(np.abs(u - v)) is the largest elementwise difference between the two
# amplitude vectors: 0.0 means the two circuits produced the identical state.
difference = np.max(np.abs(by_swap.inspect.state_vector() - by_cnots.inspect.state_vector()))
print("\nlargest difference between the two states:", difference)

Identical to the last bit. Two different circuits, the same unitary matrix.

That is a small instance of a large fact: gate sets are not unique, and a real machine implements whichever handful of gates its hardware can do well, then *compiles* everything else into them. `qsim`'s `SWAP` uses a single two-qubit tensor internally, because the point of this library is to show mechanisms rather than to be fast — but the identity above is what a compiler targeting CNOT-only hardware would use.

## 9. What you now know

- **A qubit is a unit vector in $\mathbb{C}^2$.** Its two complex components are amplitudes; $|\text{amplitude}|^2$ is a probability. That's the definition — no "both at once" required.
- **Dirac notation**: $|\psi\rangle$ is a column vector, $\langle\psi|$ its conjugate transpose, $\langle\phi|\psi\rangle$ the inner product. `qc.inspect.ket()` and `.bra()` print them, and printing them side by side shows the conjugation.
- **Gates are unitary matrices**, applied by matrix-vector multiplication. Unitary means length-preserving, which means total-probability-preserving, which is why `norm()` never moves off 1. Every single-qubit gate is a **rotation of the Bloch sphere**.
- **The Born rule** is the bridge from vector to laboratory: $\Pr[b] = |\text{amplitude of } b|^2$. It is where all quantum randomness enters, and it is a postulate.
- **Relative phase is invisible to measurement and decisive for the outcome.** $|{+}\rangle$ and $|{-}\rangle$ have identical statistics; put them each through one more $H$ and they give opposite answers with certainty. Amplitudes are complex because we need quantities that can **cancel** — and destructive interference is what every quantum algorithm is built out of.
- **Measurement collapses.** It reports an outcome *and* rewrites the state to match, destroying the superposition irreversibly. Measuring twice gives the same answer twice.
- **Many qubits combine by tensor product**, stored as an array of shape `(2,) * n` in which the axes *are* the qubits, and `psi[i, j, ...]` is the amplitude of $|ij\ldots\rangle$. Qubit 0 is the most significant bit.
- **The `inspect` namespace is the cheat namespace**: amplitudes, exact probabilities, Bloch vectors and non-collapsing sampling are all things no real quantum computer would ever hand you.

### What's next

Section 8 ended on an unanswered question: a two-qubit state has four amplitudes, but a pair of independent qubits has only two-plus-two parameters. The counting says some two-qubit states cannot be written as $|\phi\rangle \otimes |\chi\rangle$ at all.

Those states exist, they are easy to make — one $H$ and one `CNOT` — and they are called **entangled**. In such a state the *pair* has a perfectly definite description while neither qubit individually has a state at all: ask for one qubit's Bloch vector and you get an arrow of length zero, sitting at the center of the sphere with nothing to say.

**→ `02-entanglement.ipynb`: states that aren't made of parts.**